# 40 — Retrieval-Augmented Generation

### Why this exists
RAG has two quality boundaries: retrieval and generation. This lab measures them separately so a fluent answer cannot hide a retrieval failure.

### Learning objectives
retrieve documents; build grounded context; generate deterministic answers; score retrieval separately

### Prerequisite
Python dictionaries and sets

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/40-retrieval-rag.svg)

> **Question to carry through the lab:** If the right answer is absent from context, can the generator be “good enough” to fix retrieval?

## Capability contract

**Capability:** C08 — LLM Systems  
**Workstream:** AI Support Platform  
**Primary roles:** AI Engineer|ML Engineer  
**You will prove:** Build or inspect a grounded AI mechanism


## Simulation → reality bridge

**Real-system bridge:** Retrieval-augmented generation

**What this simulation proves:** It isolates the mechanism under study so you can predict and measure it without infrastructure noise.

**What the simulation hides:** The notebook uses a deterministic retriever and generator to expose grounding and retrieval failure. Real systems add chunking, embedding/index choices, model variability, context limits and citation errors.

**Transfer challenge:** Create a retrieval-vs-generation error taxonomy and define an evaluation that can attribute failures to the correct boundary.

**Production boundary:** Do not describe the toy implementation as the production technology. Before shipping, identify the real-system evidence, operational controls, and failure modes that remain untested here.


## Mental model
A RAG system is a pipeline: query → retrieval → context selection → generation → grounding/validation. Retrieval recall is upstream of generation correctness.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S19 — 1210: Support answers cite the wrong policy** → [`../work_simulations/19_acme_commerce_rag_retrieval/README.md`](../work_simulations/19_acme_commerce_rag_retrieval/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
corpus={
 "refund_policy":"Refunds are allowed within 30 days with proof of purchase.",
 "shipping_policy":"Standard shipping takes 3–5 business days.",
 "returns_contact":"Contact support for return authorization."
}
def retrieve(query,k=2):
    terms=set(query.lower().split())
    scored=sorted(((doc,len(terms & set(text.lower().split()))) for doc,text in corpus.items()),key=lambda x:x[1],reverse=True)
    return [x[0] for x in scored[:k] if x[1]>0]
print(retrieve("refund within days"))

In [ ]:
def fake_generate(query,docs):
    context=" ".join(corpus[d] for d in docs)
    if "refund" in query.lower() and "30 days" in context: return "Refunds are allowed within 30 days."
    return "Insufficient grounded evidence."
q="Can I get a refund in 20 days?"; docs=retrieve(q); print(docs,fake_generate(q,docs))

In [ ]:
# Measure retrieval recall against a tiny gold set.
gold={"refund_policy"}
print("retrieval_recall",len(gold & set(docs))/len(gold))

In [ ]:
# Break retrieval by removing the relevant source.
broken={k:v for k,v in corpus.items() if k!="refund_policy"}
old=corpus; corpus=broken
docs2=retrieve(q); print("bad retrieval",docs2,fake_generate(q,docs2))
corpus=old

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production RAG needs retrieval evaluation, citation/grounding checks, freshness, access controls, prompt-injection defenses and end-to-end monitoring.

## 9. Independent challenge
Add a distractor that ranks above the gold document. Improve retrieval without changing the generator, then improve the generator without changing retrieval. Compare which change moved which metric.

### Evidence to keep
retrieval set, generated answer, retrieval recall, and isolated retrieval-vs-generation failure

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.